# Import

In [1]:
import pandas as pd

# Charger le fichier CSV avec le bon séparateur
df = pd.read_csv("../../data/dataset.csv", sep="\\|\\|\\|", engine="python")

# Ne conserver que les colonnes utiles
df = df[['modern', 'old_french']].dropna()

# Nettoyage basique (optionnel mais recommandé)
df['modern'] = df['modern'].str.strip()
df['old_french'] = df['old_french'].str.strip()

# Vérification
print(df.sample(3))

                                                modern  \
690  Salut mes chéris ! 🌟 C'est EmmaLifestyle ici, ...   
944  Salut tout le monde ! 🎨✨ Ici Clara, votre étud...   
645  Objet : 🌟 Technologie et Soleil : Le duo parfa...   

                                            old_french  
690  Saluz mes biaulx amis ! 🌟 C'est EmmaLifestyle ...  
944  Salutations à tous ceulx qui lisent ! 🎨✨ Me no...  
645  Objet : Technologie et Soleil : Le duete parfa...  


# Vieux truc avec cartes graphique pas opti

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small", use_fast=False)

/home/utilisateur/Documents/Simplon/projet_nlp/EULA-vaaag-/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [3]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch

class TranslationDataset(Dataset):
    def __init__(self, modern_texts, old_french_texts, tokenizer, max_length=128):
        self.modern_texts = modern_texts
        self.old_french_texts = old_french_texts
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.modern_texts)
    
    def __getitem__(self, idx):
        modern = self.modern_texts[idx]
        old_french = self.old_french_texts[idx]
        modern_text = f"translate French to OldFrench: {modern}"
        
        # Tokenisation
        inputs = self.tokenizer(
            modern_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        targets = self.tokenizer(
            old_french, 
            max_length=self.max_length, 
            padding='max_length', 
            truncation=True, 
            return_tensors='pt'
        )
        
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': targets['input_ids'].flatten()
        }

# Modèle basé sur mT5 ou mBERT
model = AutoModelForSeq2SeqLM.from_pretrained('google/mt5-small')

2025-06-19 10:18:18.498763: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750321098.510617  125526 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750321098.514160  125526 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750321098.523035  125526 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750321098.523044  125526 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750321098.523045  125526 computation_placer.cc:177] computation placer alr

In [4]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
test_df, val_df = train_test_split(val_df, test_size=0.5, random_state=42)

In [10]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=True,
)

In [11]:
train_dataset = TranslationDataset(
    modern_texts=train_df["modern"].tolist(),
    old_french_texts=train_df["old_french"].tolist(),
    tokenizer=tokenizer,
    max_length=64
)

val_dataset = TranslationDataset(
    modern_texts=val_df["modern"].tolist(),
    old_french_texts=val_df["old_french"].tolist(),
    tokenizer=tokenizer,
    max_length=64
)


In [12]:
from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer
import nltk
import numpy as np
from typing import Dict

def evaluate_model(predictions, references):
    # BLEU Score
    bleu = corpus_bleu(predictions, [references])
    
    # ROUGE Score
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'])
    rouge_scores = [scorer.score(ref, pred) for ref, pred in zip(references, predictions)]
    
    # Similarité de caractères (important pour l'ancien français)
    char_similarity = [
        len(set(ref) & set(pred)) / len(set(ref) | set(pred))
        for ref, pred in zip(references, predictions)
    ]
    
    return {
        'bleu': bleu.score,
        'rouge1': np.mean([s['rouge1'].fmeasure for s in rouge_scores]),
        'char_similarity': np.mean(char_similarity)
    }

def compute_metrics(eval_pred) -> Dict:
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    return evaluate_model(decoded_preds, decoded_labels)

In [13]:
print(torch.cuda.get_device_name(0)) 

Quadro T1000


In [16]:
!cuda

zsh:1: command not found: cuda


In [14]:
model.gradient_checkpointing_enable()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipykernel_125526/3141757259.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


OutOfMemoryError: CUDA out of memory. Tried to allocate 490.00 MiB. GPU 0 has a total capacity of 3.80 GiB of which 273.69 MiB is free. Including non-PyTorch memory, this process has 2.42 GiB memory in use. Of the allocated memory 2.25 GiB is allocated by PyTorch, and 89.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Exemple d'entrée
modern_text = "Salut tout le monde ! Aujourd'hui on a fait un pique-nique dans le jardin. Après j'ai été malade, je me suis vidé par tous les trous..."
input_text = f"translate French to OldFrench: {modern_text}"

# Tokenisation
inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)

# Génération (inférence)
with torch.no_grad():
    output_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

# Décodage
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("📝 Texte d'origine :", modern_text)
print("🏰 Traduction en vieux français :", output_text)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument index in method wrapper_CUDA__index_select)